# TASK 2
## Roland Gulbinovič

## Preparation

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install tf-keras

In [ ]:
!pip install faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import requests
from bs4 import BeautifulSoup
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json


## Define functions

In [159]:
import requests
from bs4 import BeautifulSoup

# Gathers the specificed number of business articles from english version of Delfi.lt
def get_article_links(base_url, num_limit = 5000):
    links = set()
    page = 1

    while len(links) < num_limit:
        url = f"{base_url}?page={page}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        for a in soup.select('a[href*="/en/business"]'):
            href = a['href']
            if href.startswith('/'):
                href = f"https://www.delfi.lt{href}"
            links.add(href)

        print(f"Page {page}: collected {len(links)} links...")
        page += 1

    return list(links)[:5000]

# Scrapes the given article
def scrape_article(url):
    try:
        res = requests.get(url)
        soup = BeautifulSoup(res.text, 'html.parser')
        title = soup.find('h1').text.strip()
        paragraphs = soup.find_all('p')
        content = ' '.join(p.text.strip() for p in paragraphs)
        return {'url': url, 'title': title, 'content': content}
    except:
        return None

In [140]:
# Cleans the given text
def clean_text(text):
    return text.replace('\n', ' ').strip()

## Data Preparation

First we need to scrape the website and gather all the text data:

In [ ]:
links = get_article_links("https://www.delfi.lt/en/business", num_limit = 5000)

articles = []
for i, url in enumerate(links, start=1):
    article = scrape_article(url)

    if article and article['content']:
        article['content'] = clean_text(article['content'])
        articles.append(article)

    if i % 100 == 0 or i == len(links):
        print(f"Processed {i}/{len(links)} articles...")

Page 1: collected 31 links...
Page 2: collected 61 links...
Page 3: collected 91 links...
Page 4: collected 121 links...
Page 5: collected 151 links...
Page 6: collected 181 links...
Page 7: collected 211 links...
Page 8: collected 241 links...
Page 9: collected 271 links...
Page 10: collected 301 links...
Page 11: collected 331 links...
Page 12: collected 361 links...
Page 13: collected 391 links...
Page 14: collected 421 links...
Page 15: collected 451 links...
Page 16: collected 481 links...
Page 17: collected 511 links...
Page 18: collected 541 links...
Page 19: collected 571 links...
Page 20: collected 601 links...
Page 21: collected 631 links...
Page 22: collected 661 links...
Page 23: collected 690 links...
Page 24: collected 720 links...
Page 25: collected 750 links...
Page 26: collected 780 links...
Page 27: collected 810 links...
Page 28: collected 840 links...
Page 29: collected 870 links...
Page 30: collected 901 links...
Page 31: collected 931 links...
Page 32: collected 9

In [143]:
len(articles)

4736

In [144]:
with open("delfi_articles.json", "w", encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False, indent=2)

For the transformer I'm using a lightweight pre-trained transformer that is similar to Bert. It turns a sentence into a fixed-size vector (embedding) that captures its semantic meaning.

The input for the transformer is a text corpus, which has all the titles and content of the articles that we scraped earlier.

In [145]:
model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [a["title"] + ". " + a["content"] for a in articles]
doc_embeddings = model.encode(corpus, show_progress_bar=False)

Next we need to put the embeddings into a vector database using FAISS. This lets me retrieve the most relevant documents based on a query embedding.

In [146]:
dim = doc_embeddings[0].shape[0]
index = faiss.IndexFlatL2(dim)
index.add(np.array(doc_embeddings))

## Model Pipeline

Now we define our retrieve and answer generation functions.

Retrieve - we give a query - it embeds it using the same transformer as we did earlier. Then it searches the FAISS vector database for most similar articles.

Generation - Takes the retrieved articles as context and uses a pretrained LLM to generate an answer.

In [147]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    results = [articles[i] for i in indices[0]]
    return results

For the LLM I chose a random open source LLM from https://github.com/eugeneyan/open-llms. The specific one that I chose is trained to understand instructions and generate answers from context.

In [148]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
def generate_answer(context, query):
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = generator.generate(**inputs, max_length=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Then we put everything into one function/pipeline. 
Retrieve articles -> gather context -> generate answer.

In [149]:
def rag_pipeline(query, top_k=5):
    retrieved = retrieve(query, top_k=top_k)
    context = "\n\n".join([r["title"] + ": " + r["content"][:300] for r in retrieved])
    return generate_answer(context, query)

In [156]:
answer = rag_pipeline("")